## Задание 1. Разработка и тестирование определения тональности текста

### Шаг 1. Загрузка и подготовка данных

In [16]:
from pathlib import Path
from datasets import load_dataset
import pandas as pd


DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

SPLITS = [
    "train",
    "test",
    "validation"
]


def save_data(dataset):
    for split_name, split in dataset.items():
        if not (DATA_DIR / f"{split_name}.csv").exists():
            split.to_csv(DATA_DIR / f"{split_name}.csv", index=False)
            print(split_name, len(split))


if all((DATA_DIR / f"{t}.csv").exists() for t in SPLITS):
    print("Данные уже загружены")
else:
    ds = load_dataset("jbeno/sentiment_merged")
    save_data(ds)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
validation_df = pd.read_csv(DATA_DIR / "validation.csv")

df_columns = train_df.columns
head = train_df.head()

print(df_columns)
print(head)

for split, df in zip(SPLITS, [train_df, test_df, validation_df]):
    print('-' * 50)
    print(f"{split.upper()}")
    print(f"Shape: {df.shape}")

    print("\nРаспределение классов:")
    print(df["label"].value_counts(normalize=True))

    print("\nПропуски:")
    print(df.isna().sum())
    print()



Данные уже загружены
Index(['sentence', 'label', 'source', 'split'], dtype='str')
                                            sentence     label       source  \
0  Those 2 drinks are part of the HK culture and ...  negative  dynasent_r2   
1  I was told by the repair company that was doin...  negative  dynasent_r1   
2             It is there to give them a good time .   neutral    sst_local   
3  Like leafing through an album of photos accomp...  negative    sst_local   
4         Johnny was a talker and liked to have fun.  positive  dynasent_r1   

   split  
0  train  
1  train  
2  train  
3  train  
4  train  
--------------------------------------------------
TRAIN
Shape: (102097, 4)

Распределение классов:
label
neutral     0.481385
positive    0.304015
negative    0.214600
Name: proportion, dtype: float64

Пропуски:
sentence    0
label       0
source      0
split       0
dtype: int64

--------------------------------------------------
TEST
Shape: (6530, 4)

Распределение классо

### Шаг 2. Предобработка текстов

In [22]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

1. Токенизация (разделение на слова).
2. Приведение к нижнему регистру.
3. Удаление стоп-слов.
4. Лемматизация (нормализация слов)

In [23]:
for sentence in train_df["sentence"].head():
    print(sentence)
    tokens = word_tokenize(sentence)
    print(tokens)
    lower = sentence.lower()
    print(lower)

    stop_words = set(nltk.corpus.stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
    print(filtered_tokens)

    lemmatizer = nltk.stem.WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]
    print(lemmatized_tokens)

    print('-' * 50)

Those 2 drinks are part of the HK culture and has years of history. It is so bad.
['Those', '2', 'drinks', 'are', 'part', 'of', 'the', 'HK', 'culture', 'and', 'has', 'years', 'of', 'history', '.', 'It', 'is', 'so', 'bad', '.']
those 2 drinks are part of the hk culture and has years of history. it is so bad.
['2', 'drinks', 'part', 'HK', 'culture', 'years', 'history', '.', 'bad', '.']
['2', 'drink', 'part', 'HK', 'culture', 'year', 'history', '.', 'bad', '.']
--------------------------------------------------
I was told by the repair company that was doing the car repair that fixing the rim was "impossible" and to replace it.
['I', 'was', 'told', 'by', 'the', 'repair', 'company', 'that', 'was', 'doing', 'the', 'car', 'repair', 'that', 'fixing', 'the', 'rim', 'was', '``', 'impossible', "''", 'and', 'to', 'replace', 'it', '.']
i was told by the repair company that was doing the car repair that fixing the rim was "impossible" and to replace it.
['told', 'repair', 'company', 'car', 'repair'

### Шаг 3. Преобразование текстов в векторы
- Bag-of-Words или TF-IDF.
- Для лексиконного метода — вычислите «тональный балл» по словарю.